# 1) Import Configuration and Functions:

In [0]:
%run ../common/configuration


In [0]:
%run ../common/functions

## 2) Define Constructor Standings Schema:

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

constructor_standings_schema = StructType([
    StructField("season", IntegerType(), False),
    StructField("round", IntegerType(), True),
    StructField("position", IntegerType(), True),
    StructField("position_text", StringType(), True),
    StructField("points", FloatType(), True),
    StructField("wins", IntegerType(), True),
    StructField("constructor_id", StringType(), False),
    StructField("constructor_name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True),
])

constructor_standings_input_path = f"{processed_folder_path}/constructor_standings/csv/constructor_standings.csv"

constructor_standings_df = spark.read \
    .option("header", True) \
    .schema(constructor_standings_schema) \
    .csv(constructor_standings_input_path)

constructor_standings_dropped_df = constructor_standings_df.drop("url", "constructor_name", "nationality")

# 3) Transform Constructor Standings Data:

The steps included:

- Fill Null (or "-") Position and Position Text with "0".
- Create Surrogate Key.
- Add Data Source and File Date.

In [0]:
from pyspark.sql.functions import lit, when, col

constructor_standings_with_audit_df = constructor_standings_dropped_df \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

constructor_standings_date_df = add_ingestion_date(constructor_standings_with_audit_df)
constructor_standings_position_filled_df = constructor_standings_date_df \
    .fillna(0, subset=["position"]) \
    .withColumn(
        "position_text",
        when(col("position_text") == "-", "0").otherwise(col("position_text"))
    )

constructor_standings_fill_df = constructor_standings_position_filled_df.fillna("None")

constructor_standings_final_df = add_surrogate_key(
    constructor_standings_fill_df,
    key_column_name="constructor_standing_sk",
    hash_columns=["season", "round", "position", "position_text", "points", "wins",
                  "constructor_id"],
)

print("Final columns going into the write:", constructor_standings_final_df.columns)

# 4) Save the Processed Dataset to Delta Lake:

In [0]:
constructor_standings_output_path = f"{processed_folder_path}/constructor_standings/delta"

spark.sql("CREATE DATABASE IF NOT EXISTS f1_processed")

upsert_if_changed_for_seasons(
    input_df=constructor_standings_final_df,
    db_name="f1_processed",
    table_name="constructor_standings",
    output_path=constructor_standings_output_path,
    merge_key_columns=["season", "constructor_id"],
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(constructor_standings_output_path))

In [0]:
build_presentation_fact(
    processed_location=f"{processed_folder_path}/constructor_standings/delta",
    presentation_directory=f"{presentation_folder_path}/fact_constructor_standings/delta",
    db_name="f1_presentation",
    table_name="fact_constructor_standings",
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(f"{presentation_folder_path}/fact_constructor_standings/delta"))

# 5) Save backup Constructor Standings in CSV format:

In [0]:
import io
import csv

constructor_standings_backup_path = f"{presentation_folder_path}/fact_constructor_standings/csv/fact_constructor_standings.csv"

backup_rows = [row.asDict() for row in constructor_standings_final_df.collect()]
backup_fieldnames = constructor_standings_final_df.columns

backup_buffer = io.StringIO()
backup_writer = csv.DictWriter(backup_buffer, fieldnames=backup_fieldnames)
backup_writer.writeheader()
backup_writer.writerows(backup_rows)

dbutils.fs.put(constructor_standings_backup_path, backup_buffer.getvalue(), overwrite=True)
print(f"backup saved: {constructor_standings_backup_path}")